In [1]:
import polars as pl

data = pl.read_parquet("../data/smoothcomp/tiny/results.parquet")

In [2]:
belts = {
    "White": 0,
    "Blue": 1,
    "Purple": 2,
    "Brown": 3,
    "Black": 4
}

absolute_df = (
    data
    .filter(pl.col("is_absolute"))
    .filter(pl.col("loser_inferred_belt").is_in(belts) & pl.col("winner_inferred_belt").is_in(belts))
    .select(pl.col("winner_inferred_belt"), pl.col("loser_inferred_belt"))
)

In [ ]:
import numpy as np

num_matches_matrix = np.zeros((len(belts), len(belts)), dtype=np.int32)
higher_belt_wins_matrix = np.zeros((len(belts), len(belts)), dtype=np.float32)

for lower_belt, lower_belt_idx in belts.items():
    for higher_belt, higher_belt_idx in list(belts.items())[lower_belt_idx:]:
        lower_belt_wins = (pl.col("winner_inferred_belt") == lower_belt) & (pl.col("loser_inferred_belt") == higher_belt)
        higher_belt_wins = (pl.col("loser_inferred_belt") == lower_belt) & (pl.col("winner_inferred_belt") == higher_belt)

        curr_df = absolute_df.filter(lower_belt_wins | higher_belt_wins)
        curr_num_matches = len(curr_df)
        num_matches_matrix[higher_belt_idx, lower_belt_idx] = curr_num_matches

        if higher_belt_idx == lower_belt_idx:
            continue
        elif len(curr_df) == 0:
            higher_belt_wins_matrix[higher_belt_idx, lower_belt_idx] = -1.0
            

        num_higher_belt_wins = len(absolute_df.filter(higher_belt_wins))
        higher_belt_win_proportion = num_higher_belt_wins / curr_num_matches

In [7]:
num_matches

array([[58,  0,  0,  0,  0],
       [10,  4,  0,  0,  0],
       [ 5,  0,  8,  0,  0],
       [ 0,  0,  0,  5,  0],
       [ 0,  0,  0,  0, 11]], dtype=int32)